In [1]:
import json
import random
import shutil
import os
from pathlib import Path
from tqdm import tqdm

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# Đường dẫn gốc (Thay đổi cho khớp với máy của bạn)
BASE_FOLDER = Path('../data/viet-cultural-vqa')
SOURCE_IMG_ROOT = BASE_FOLDER / 'images'
RAW_DATA_PATH = BASE_FOLDER / 'vietnamese_vqa_dataset.json'
INDEX_PATH = BASE_FOLDER / 'images_index.json'

# Đường dẫn đích sau khi chia
TARGET_ROOT = Path('../data/raw')
TARGET_ROOT.mkdir(parents=True, exist_ok=True)
# Danh mục cần lọc (Có thể thay đổi để lấy các category khác nhau)
filter_category = ['am_thuc', 'trang_phuc', 'kien_truc', 'le_hoi', 'nhac_cu', 'the_thao_truyen_thong']


def process_complete_split():
    # 1. Load dữ liệu gốc
    print("--- Đang tải dữ liệu ---")
    with open(INDEX_PATH, 'r', encoding='utf-8') as f:
        index_data = json.load(f)
    with open(RAW_DATA_PATH, 'r', encoding='utf-8') as f:
        raw_dataset = json.load(f)

    # Map image_path vào dataset để truy xuất nhanh
    # --- SỬA LỖI TẠI ĐÂY: Tạo khóa duy nhất dựa trên đường dẫn ---
    dataset_dict = {}
    for item in raw_dataset:
        path = item.get('image_path', '')
        # Trích xuất phần 'category/subcategory/filename' từ 'data/images/category/subcategory/filename'
        # Ví dụ: 'am_thuc/banh_chung_Tet/000001.jpg'
        parts = Path(path).parts
        if len(parts) >= 3:
            unique_key = "/".join(parts[-3:]) # Lấy 3 phần cuối cùng của đường dẫn
            dataset_dict[unique_key] = item
    
    # Khởi tạo các thùng chứa dữ liệu
    split_records = {
        'train': [],
        'test': [],
        'val': []
    }
    knowledge_base = []
    seen_kb_content = set()

    # 2. Duyệt qua từng Category và Subcategory để chia công bằng
    # Chỉ lấy những category đã lọc trước đó (filter_category) để đảm bảo tập trung vào các danh mục chính
    categories = index_data.get('categories', {})
    for cat_name, cat_info in categories.items():
        if cat_name not in filter_category:
            continue
        print(f"\nĐang xử lý danh mục: {cat_name}")
        subcategories = cat_info.get('subcategories', {})
        
        for sub_name, sub_info in subcategories.items():
            files = sub_info.get('files', [])
            random.seed(42) # Đảm bảo tính tái lập
            random.shuffle(files)

            # Chia tỉ lệ 8/1/1
            total = len(files)
            tr_idx = int(total * 0.8)
            te_idx = int(total * 0.9)

            file_splits = {
                'train': files[:tr_idx],
                'test': files[tr_idx:te_idx],
                'val': files[te_idx:]
            }

            for split_name, file_list in file_splits.items():
                # Tạo folder ảnh vật lý tương ứng: data/raw/images_train/am_thuc/mon_an/
                dest_img_dir = TARGET_ROOT / f"images_{split_name}" / cat_name / sub_name
                dest_img_dir.mkdir(parents=True, exist_ok=True)

                for filename in tqdm(file_list, desc=f"Split {split_name} - {sub_name}"):
                    lookup_key = f"{cat_name}/{sub_name}/{filename}"
                    if lookup_key not in dataset_dict:
                        continue
                    
                    item = dataset_dict[lookup_key]
                    img_id = filename.split('.')[0]
                    
                    # A. COPY ẢNH VẬT LÝ
                    src_path = SOURCE_IMG_ROOT / cat_name / sub_name / filename
                    if src_path.exists():
                        shutil.copy2(src_path, dest_img_dir / filename)
                    
                    # B. PHẲNG HÓA DỮ LIỆU JSONL
                    img_analysis = item.get('image_analysis', {})
                    # Tạo Vision Caption xịn cho đồ án
                    vision_caption = (
                        f"{img_analysis.get('overall_description', '')} "
                        f"Đối tượng trong ảnh: {', '.join(img_analysis.get('main_objects', []))}. "
                        f"Bố cục: {img_analysis.get('visual_details', {}).get('composition', '')}"
                    ).strip()
                    # Mỗi câu hỏi sẽ có giải thích chi tiết riêng
                    # Đường dẫn ảnh mới tương đối để model dễ nạp
                    new_img_path = f"images_{split_name}/{cat_name}/{sub_name}/{filename}"
                    # Tạo cultural knowledge content cho tập train
                    cultural = item.get('cultural_context', {})
                        # Lọc trùng symbolism của từng câu hỏi trong cùng 1 ảnh
                        # Nếu chỉ có 1 symbolism thì vẫn lấy, nếu có nhiều symbolism thì chỉ lấy unique để tránh trùng lặp thông tin
                        # Lọc trùng symbolism và loại bỏ giá trị None/Trống
                        # Lọc trùng symbolism và xử lý cả trường hợp là chuỗi hoặc danh sách
                    raw_syms = []
                    for que in item.get('questions', []):
                        context = que.get('additional_context')
                        if context:
                            sym = context.get('symbolism')
                            if sym is not None:
                                # Nếu là list, gộp các phần tử thành chuỗi
                                if isinstance(sym, list):
                                    sym = ". ".join([str(s) for s in sym if s is not None])
                                # Nếu là chuỗi, giữ nguyên
                                if isinstance(sym, str) and sym.strip():
                                    raw_syms.append(sym.strip())

                    # Bây giờ mới dùng set để lọc trùng các chuỗi đã chuẩn hóa
                    unique_syms = list(set(raw_syms))
                    knowledge_content = (
                        f"Lịch sử: {cultural.get('historical_context', '')}. "
                        f"Ý nghĩa: {' '.join(unique_syms)}. "
                        f"Vùng miền: {cultural.get('regional_significance', '')}"
                    ).strip()
            
                    for q in item.get('questions', []):
                        # Ghép answer_label từ câu trả lời và giải thích chi tiết để tăng độ phong phú cho mô hình học
                        # Lấy đúng answer và explanation của từng câu hỏi q
                        current_ans = q.get('answer', '')
                        current_ext = q.get('detailed_explanation', '')
                        new_answer_label = f"{current_ans}. {current_ext}".strip()
                        split_records[split_name].append({
                            "id": f"{cat_name}|{sub_name}|{img_id}|q{q.get('question_id', '')}",
                            "image_path": new_img_path,
                            "category": cat_name,
                            "subcategory": sub_name,
                            "keyword": item.get('keyword'),
                            "question": q.get('question'),
                            "standalone_question": "", # rỗng để chèn module sau
                            "cultural_context": knowledge_content, # gộp tất cả tri thức văn hóa vào 1 trường để dễ truy xuất kèm symbolism đã được lọc trùng
                            "vision_caption": vision_caption, # nên thêm caption vào để tăng cường thông tin hình ảnh cho mô hình
                            "answer_label": new_answer_label, # gộp answer và giải thích chi tiết
                            "rewrite_method": "" # rỗng để chèn module sau
                        })

                    # C. GOM TỰ TRI THỨC CHO RAG (Chỉ lấy từ tập Train)
                    if split_name == 'train':
                        if knowledge_content not in seen_kb_content and len(knowledge_content) > 20:
                            knowledge_base.append({
                                "keyword": item.get('keyword'),
                                "content": knowledge_content,
                                "category": cat_name
                            })
                            seen_kb_content.add(knowledge_content)

    # 3. XUẤT FILE KẾT QUẢ
    print("\n--- Đang xuất file JSONL ---")
    for split_name in ['train', 'test', 'val']:
        output_file = TARGET_ROOT / f"{split_name}.jsonl"
        with open(output_file, 'w', encoding='utf-8') as f:
            for record in split_records[split_name]:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
    
    # Xuất Knowledge Base cho RAG
    with open(TARGET_ROOT / "knowledge_base.jsonl", 'w', encoding='utf-8') as f:
        for entry in knowledge_base:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"\nHOÀN THÀNH!")
    print(f"Dữ liệu Train: {len(split_records['train'])} câu hỏi")
    print(f"Dữ liệu Val: {len(split_records['val'])} câu hỏi")
    print(f"Dữ liệu Test: {len(split_records['test'])} câu hỏi")
    print(f"Knowledge Base: {len(knowledge_base)} đoạn tri thức")
    print(f"Tất cả ảnh và metadata đã nằm trong: {TARGET_ROOT}")

if __name__ == "__main__":
    process_complete_split()

--- Đang tải dữ liệu ---

Đang xử lý danh mục: am_thuc


Split val - giò_thu: 100%|██████████| 6/6 [00:00<00:00, 348.56it/s]



Đang xử lý danh mục: le_hoi


Split val - tet_Han_thực: 100%|██████████| 5/5 [00:00<00:00, 106.74it/s]



Đang xử lý danh mục: nhac_cu


...


Đang xử lý danh mục: trang_phuc


Split val - nón_quai_thao: 100%|██████████| 6/6 [00:00<00:00, 36.41it/s]



Đang xử lý danh mục: the_thao_truyen_thong


Split val - vật_cổ_truyen_Việt_Nam: 100%|██████████| 6/6 [00:00<00:00, 51.26it/s]



--- Đang xuất file JSONL ---

HOÀN THÀNH!
Dữ liệu Train: 51918 câu hỏi
Dữ liệu Val: 7161 câu hỏi
Dữ liệu Test: 6403 câu hỏi
Knowledge Base: 10719 đoạn tri thức
Tất cả ảnh và metadata đã nằm trong: ..\data\raw


In [ ]:
from PIL import Image
import os
from pathlib import Path
from tqdm import tqdm

# Cấu hình folder
# Nguồn: ../data/raw/images_train, images_test, images_val
# Đích: ../data/raw_resized/images_train, ...
BASE_RAW = Path('../data/raw')
DEST_RAW = Path('../data/raw_resized')

def process_nested_resize(split_name, size=(224, 224)):
    src_root = BASE_RAW / f"images_{split_name}"
    dst_root = DEST_RAW / f"images_{split_name}"
    
    # Tìm tất cả file ảnh ở mọi cấp độ sâu (cat/subcat/...)
    # rglob('*.*') sẽ tìm hết, không quan trọng lồng bao nhiêu cấp
    image_files = [p for p in src_root.rglob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    
    print(f"\n--- Đang xử lý tập {split_name}: {len(image_files)} ảnh ---")
    
    for src_path in tqdm(image_files):
        try:
            # Tạo đường dẫn đích tương ứng
            # Ví dụ: src là images_train/am_thuc/pho/1.jpg
            # -> relative là am_thuc/pho/1.jpg
            relative_path = src_path.relative_to(src_root)
            dst_path = dst_root / relative_path
            
            # Tự động tạo folder cat và subcat nếu chưa có
            dst_path.parent.mkdir(parents=True, exist_ok=True)
            
            with Image.open(src_path) as img:
                img = img.convert('RGB')
                img = img.resize(size, Image.Resampling.LANCZOS)
                # Lưu chất lượng 90 để ảnh vẫn nét mà dung lượng cực nhẹ
                img.save(dst_path, "JPEG", quality=90)
                
        except Exception as e:
            print(f"\nLỗi tại {src_path.name}: {e}")

# Chạy cho 3 tập
if __name__ == "__main__":
    for split in ['train', 'test', 'val']:
        process_nested_resize(split)
    print("\n--- XONG! Chi check folder ../data/raw_resized nhé ---")